In [1]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import polars as pl
import pandas as pd
import json
import os
import sys

sys.path.append("..")

import seaborn as sns

sns.set()
from settings import (
    random_state,
    PROJECT_PATH,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
)
from sklearn.ensemble import RandomForestClassifier

In [2]:
transactions = pl.read_parquet(
    os.path.join(PROJECT_PATH, "real_estate_transactions_engineered.parquet")
)

X = transactions.drop([REGRESSION_TARGET, CLASSIFICATION_TARGET])
y_regression = transactions[REGRESSION_TARGET]
y_classification = transactions[CLASSIFICATION_TARGET]


with open("../features_used.json", "r") as f:
    feature_names = json.load(f)

with open("../categorical_features_used.json", "r") as f:
    categorical_features = json.load(f)


numerical_features = [col for col in feature_names if col not in categorical_features]

In [3]:
classifier = RandomForestClassifier(random_state=random_state, verbose=False)

X_train, X_test, y_train, y_test = train_test_split(
    X.to_pandas(), y_classification.to_pandas(), random_state=random_state
)

classifier.fit(X_train[feature_names], y_train)


RandomForestClassifier(random_state=42, verbose=False)

In [4]:
y_train_pred = classifier.predict(X_train[feature_names])
y_test_pred = classifier.predict(X_test[feature_names])
X_train["prediction"] = y_train_pred
X_test["prediction"] = y_test_pred



In [5]:
def get_prediction_type(prediction, target):
    if prediction == 1 and target == 1:
        return "true_positive"
    elif prediction == 0 and target == 0:
        return "true_negative"
    elif prediction == 1 and target == 0:
        return "false_positive"
    elif prediction == 0 and target == 1:
        return "false_negative"
    else:
        return "unknown"

In [6]:
X_train[CLASSIFICATION_TARGET] = y_train
X_train["prediction_type"] = X_train.apply(
    lambda row: get_prediction_type(row["prediction"], row[CLASSIFICATION_TARGET]),
    axis=1,
)

X_test[CLASSIFICATION_TARGET] = y_test
X_test["prediction_type"] = X_test.apply(
    lambda row: get_prediction_type(row["prediction"], row[CLASSIFICATION_TARGET]),
    axis=1,
)

An interesting approach for establishing the link between model errors and quantitative features consists of comparing:
* The distribution of a feature within the training or test set
* The distribution of a feature among observations that were misclassified by the model (for example, false positives or false negatives)
* Repeat the process for the other features

This would help detect whether there are "typical" values for certain features where the model struggles particularly with its classification!


In [7]:
# This function returns the observations for the groups we want to study
def get_data_group(X, group_label):
    if group_label == "all":
        X_group = X.copy()
    if group_label == "false_positive":
        X_group = X[X["prediction_type"] == "false_positive"]
    if group_label == "false_negative":
        X_group = X[X["prediction_type"] == "false_negative"]
    if group_label == "true_positive":
        X_group = X[X["prediction_type"] == "true_positive"]
    if group_label == "true_negative":
        X_group = X[X["prediction_type"] == "true_negative"]
    return X_group

In [8]:
# We compare two groups of our choice
def compare_all_features_distribution_groups(X, group_1, group_2, numerical_features):

    """
    The following 5 lines of code prepare the data to leverage
    the displot function and its "hue" argument, which handles as many distributions as there are groups to compare.
    Each group must belong to the same column passed to "hue".
    """
    X_group_1 = get_data_group(X, group_1)
    X_group_1["group"] = group_1
    X_group_2 = get_data_group(X, group_2)
    X_group_2["group"] = group_2
    X_plot = pd.concat([X_group_1, X_group_2], axis = 0)

    # Since we are plotting distributions, including categorical features makes no sense
    for feature in numerical_features:
        sns.displot(data = X_plot, x=feature, hue="group")
        plt.xlabel("Feature Distribution")
        plt.title("Comparing feature distribution for {} and {}".format(group_1, group_2))

In [ ]:
compare_all_features_distribution_groups(X_test, "false_negative", "all", numerical_features)

In [ ]:
compare_all_features_distribution_groups(X_test, "false_positive", "all", numerical_features)

Regarding false positives and false negatives, it is difficult to draw any firm conclusions. The distributions of false positives and false negatives closely follow the distributions of the full test set.


We could push the analysis further and focus on other groups:
* All false negatives with a predicted probability below 10%: these are the observations where the model is particularly wrong, as it does not even consider them a borderline case between the two classes.
* All false positives with a very high predicted probability (e.g. 90%): these are the observations where the model is excessively confident, even though it should not be.
